# 03 — Model Evaluation

**Belgium Campus · Student Academic Risk Prediction**

> **How well does the saved model predict academic risk — and where does it fail?**

---
## 1. Project Overview

| Item | Detail |
|------|--------|
| Task | `Low` / `Medium` / `High` risk |
| Model | `models/trained_model.pkl` |
| Holdout | Stratified 20%, `random_state=42` |

---
## 2. Import Libraries

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_recall_fscore_support, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.2
RISK_ORDER = ["Low", "Medium", "High"]
RISK_COLORS = {"Low": "#2a9d8f", "Medium": "#e9c46a", "High": "#e76f51"}
LEAKAGE_OR_ID_COLS = ["student_id", "profile", "risk_score", "risk_label"]
print("Libraries imported.")

---
## 3. Load Dataset

In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "data").exists() else NOTEBOOK_DIR.parent
DATA_PATH = PROJECT_ROOT / "data" / "synthetic" / "students.csv"
MODEL_DIR = PROJECT_ROOT / "models"

df = pd.read_csv(DATA_PATH)
feature_cols = [c for c in df.columns if c not in LEAKAGE_OR_ID_COLS]
X = df[feature_cols].copy()
y = df["risk_label"].map({l: i for i, l in enumerate(RISK_ORDER)}).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
_, df_test, _, _ = train_test_split(
    df, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Features: {feature_cols}")
print(f"Train {len(X_train):,} | Test {len(X_test):,}")
display(df.head(3))

---
## 4. Load Saved Model

In [ ]:
model = joblib.load(MODEL_DIR / "trained_model.pkl")
label_encoder = joblib.load(MODEL_DIR / "label_encoder.pkl")
estimator = model.named_steps["model"]
estimator_name = type(estimator).__name__
print(f"Estimator: {estimator_name}")
print(f"Classes: {list(label_encoder.classes_)}")
display(model)

---
## 5. Make Predictions

In [ ]:
y_pred = model.predict(X_test)
y_test_labels = label_encoder.inverse_transform(y_test)
y_pred_labels = label_encoder.inverse_transform(y_pred)
display(pd.DataFrame({
    "actual": y_test_labels[:10],
    "predicted": y_pred_labels[:10],
    "correct": y_test_labels[:10] == y_pred_labels[:10],
}))
print(f"Correct: {int((y_pred == y_test).sum())} / {len(y_test)}")

---
## 6. Accuracy

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
display(pd.DataFrame({
    "metric": ["accuracy"],
    "value": [round(accuracy, 4)],
    "interpretation": [f"{accuracy:.1%} of held-out students correct"],
}))

---
## 7. Precision, Recall & F1

In [ ]:
order_idx = [list(label_encoder.classes_).index(r) for r in RISK_ORDER]
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=order_idx, zero_division=0
)
per_class = pd.DataFrame({
    "risk_label": RISK_ORDER,
    "precision": np.round(precision, 3),
    "recall": np.round(recall, 3),
    "f1": np.round(f1, 3),
    "support": support,
})
macro = {
    "precision_macro": round(precision_score(y_test, y_pred, average="macro"), 4),
    "recall_macro": round(recall_score(y_test, y_pred, average="macro"), 4),
    "f1_macro": round(f1_score(y_test, y_pred, average="macro"), 4),
    "f1_weighted": round(f1_score(y_test, y_pred, average="weighted"), 4),
}
display(per_class)
display(pd.DataFrame([macro]))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
w = 0.25
ax.bar(x - w, precision, w, label="Precision", color="#457b9d")
ax.bar(x, recall, w, label="Recall", color="#2a9d8f")
ax.bar(x + w, f1, w, label="F1", color="#e9c46a")
ax.set_xticks(x, RISK_ORDER)
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title("Per-class precision / recall / F1")
plt.tight_layout()
plt.show()

---
## 8. Classification Report

In [ ]:
print(classification_report(
    y_test, y_pred, labels=order_idx, target_names=RISK_ORDER, digits=3, zero_division=0
))

---
## 9. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=order_idx)
display(pd.DataFrame(cm, index=[f"Actual {r}" for r in RISK_ORDER],
                     columns=[f"Pred {r}" for r in RISK_ORDER]))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3), RISK_ORDER)
ax.set_yticks(range(3), RISK_ORDER)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion — {estimator_name}")
thresh = cm.max() / 2 if cm.max() else 0
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

high_idx = RISK_ORDER.index("High")
high_misses = int(cm[high_idx].sum() - cm[high_idx, high_idx])
print(f"High-risk misses: {high_misses}")

---
## 10. Feature Importance

In [ ]:
preprocess = model.named_steps["preprocess"]
feature_names = preprocess.get_feature_names_out()
if hasattr(estimator, "feature_importances_"):
    importances = estimator.feature_importances_
    kind = "feature_importances_"
elif hasattr(estimator, "coef_"):
    coef = np.asarray(estimator.coef_)
    importances = np.mean(np.abs(coef), axis=0) if coef.ndim == 2 else np.abs(coef)
    kind = "mean_|coef_|"
else:
    raise AttributeError(estimator_name)

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False).reset_index(drop=True)
)
print(f"Source: {kind}")
display(importance_df.head(15))

top = importance_df.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top["feature"], top["importance"], color="#457b9d", edgecolor="white")
ax.set_title(f"Top signals — {estimator_name}")
plt.tight_layout()
plt.show()

---
## 11. Example Predictions

In [ ]:
examples = df_test.copy()
examples["actual"] = y_test_labels
examples["predicted"] = y_pred_labels
examples["correct"] = examples["actual"] == examples["predicted"]

show_cols = [
    "student_id", "programme", "specialisation", "year_of_study",
    "attendance", "assignment_completion", "test_average", "midyear_average",
    "bc_connect_activity", "missed_assessments", "failed_modules",
    "actual", "predicted",
]

print("Correct (sample):")
display(examples.loc[examples["correct"], show_cols].head(5))
print("Incorrect (sample):")
mistakes = examples.loc[~examples["correct"], show_cols]
display(mistakes.head(5) if len(mistakes) else pd.DataFrame({"note": ["None"]}))
print("High-risk misses:")
hm = examples[(examples["actual"] == "High") & (examples["predicted"] != "High")]
display(hm[show_cols].head(5) if len(hm) else pd.DataFrame({"note": ["None"]}))

---
## 12. Model Strengths & Limitations

In [ ]:
high_recall = float(per_class.loc[per_class["risk_label"] == "High", "recall"].iloc[0])
medium_f1 = float(per_class.loc[per_class["risk_label"] == "Medium", "f1"].iloc[0])
display(pd.DataFrame({
    "type": ["Strength", "Strength", "Limitation", "Limitation"],
    "point": [
        f"Test accuracy {accuracy:.1%}; High recall {high_recall:.1%}",
        "Uses BC-style predictors (attendance, completion, BC Connect, misses, fails)",
        f"Medium class often hardest (F1={medium_f1:.1%})",
        "Synthetic data — not live campus extracts",
    ],
}))

---
## 13. Conclusion

The model is a credible baseline for Belgium Campus academic-risk classification using percentage marks and engagement signals.

In [ ]:
display(pd.DataFrame({
    "item": ["estimator", "test_students", "accuracy", "f1_macro", "high_recall", "high_misses"],
    "value": [estimator_name, len(y_test), round(accuracy, 4), macro["f1_macro"],
              round(high_recall, 4), high_misses],
}))
print("Evaluation complete.")